# Project 3 — Operations Efficiency & Data Quality

**Goal.** Profile a year of fulfillment activity (6 warehouses, 5 carriers) against a 7-day promised SLA, identify bottlenecks, and stand up a data-quality scorecard for the upstream WMS feed.

**Inputs.** `shipments.csv` (clean) and `shipments_raw.csv` (intentionally messy) in `../data/`.

**Outputs.** Cleaning logic, KPI calculations, DQ rule library, and the dashboard at `../dashboard/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

DATA = Path('../data')
pd.options.display.float_format = '{:,.2f}'.format

## 1. Load both feeds

In [ ]:
clean = pd.read_csv(DATA / 'shipments.csv',
                    parse_dates=['order_date', 'ship_date', 'delivery_date'])
raw   = pd.read_csv(DATA / 'shipments_raw.csv',
                    parse_dates=['order_date', 'ship_date', 'delivery_date'])
print(clean.shape, raw.shape)

## 2. Data-quality scorecard

Each rule is one of the four classic DQ dimensions: completeness, validity, uniqueness, referential integrity.

In [ ]:
VALID_CARRIERS = {'DHL', 'FedEx', 'UPS', 'USPS', 'Local'}

def dq_scorecard(df):
    n = len(df)
    rules = {
        'rows_total'              : n,
        'missing_order_date'      : df['order_date'].isna().sum(),
        'missing_warehouse_city'  : df['warehouse_city'].isna().sum(),
        'missing_delivery_date'   : df['delivery_date'].isna().sum(),
        'negative_shipping_cost'  : (df['shipping_cost'] < 0).sum(),
        'delivery_before_order'   : (df['delivery_date'] < df['order_date']).sum(),
        'unknown_carrier'         : (~df['carrier'].isin(VALID_CARRIERS)).sum(),
        'duplicate_shipment_id'   : df.duplicated(subset=['shipment_id']).sum(),
    }
    issues = sum(v for k, v in rules.items() if k != 'rows_total')
    rules['dq_score_pct'] = round(100 - issues / n * 100, 2)
    return rules

pd.Series(dq_scorecard(raw)).to_frame('value')

## 3. Cleaning pipeline

Apply each rule in order. The clean frame should match the curated `shipments.csv`.

In [ ]:
def clean_pipeline(df):
    df = df.copy()
    df = df.drop_duplicates(subset=['shipment_id'])
    df = df[df['order_date'].notna() & df['warehouse_city'].notna()]
    df = df[df['shipping_cost'].fillna(0) >= 0]
    df = df[df['carrier'].isin(VALID_CARRIERS)]
    df = df[(df['delivery_date'].isna()) | (df['delivery_date'] >= df['order_date'])]
    return df

cleaned = clean_pipeline(raw)
print(f'Raw:   {len(raw):,} rows')
print(f'Clean: {len(cleaned):,} rows  ({(1 - len(cleaned)/len(raw))*100:.2f}% dropped)')

## 4. Operations KPIs (on the curated table)

In [ ]:
kpis = {
    'Shipments'            : len(clean),
    'On-time %'            : (clean['status'] == 'On-time').mean() * 100,
    'Late %'               : (clean['status'] == 'Late').mean() * 100,
    'Failed %'             : (clean['status'] == 'Failed').mean() * 100,
    'Avg lead time (days)' : clean['total_days'].mean(),
    'Avg processing (days)': clean['processing_days'].mean(),
    'Avg transit (days)'   : clean['transit_days'].mean(),
    'Avg shipping cost'    : clean['shipping_cost'].mean(),
}
pd.Series(kpis).to_frame('value').round(2)

## 5. Warehouse SLA (where bottlenecks live)

In [ ]:
wh = (clean.groupby(['warehouse_id', 'warehouse_city'])
             .agg(shipments=('shipment_id', 'count'),
                  on_time_pct=('status', lambda s: (s == 'On-time').mean() * 100),
                  avg_processing=('processing_days', 'mean'),
                  avg_transit=('transit_days', 'mean'),
                  avg_cost=('shipping_cost', 'mean'))
             .reset_index()
             .sort_values('on_time_pct'))
wh.round(2)

## 6. Carrier scorecard

In [ ]:
car = (clean.groupby('carrier')
              .agg(shipments=('shipment_id', 'count'),
                   on_time_pct=('status', lambda s: (s == 'On-time').mean() * 100),
                   failed_pct =('status', lambda s: (s == 'Failed').mean() * 100),
                   avg_transit=('transit_days', 'mean'),
                   avg_cost   =('shipping_cost', 'mean'))
              .reset_index()
              .sort_values('on_time_pct', ascending=False))
car.round(2)

## 7. Bottleneck attribution

Decompose lead time into processing vs transit and look at p95 to surface the long tail.

In [ ]:
stages = pd.DataFrame({
    'stage'    : ['Processing', 'Transit'],
    'mean'     : [clean['processing_days'].mean(), clean['transit_days'].mean()],
    'p95'      : [clean['processing_days'].quantile(0.95),
                  clean['transit_days'].quantile(0.95)],
    'std'      : [clean['processing_days'].std(),
                  clean['transit_days'].std()],
})
stages.round(2)

See `../report.md` for the executive write-up.